# Tutorial 03: Inference and Evaluation

In this final tutorial, we use our **YAML configuration** to ensure inference parameters match training, and evaluate our model using the **strict testing file list**.

## 1. Setup and Imports

In [1]:
import os
import sys
import yaml
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath('../../src'))

from bioacoustica.testing.predictor import Predictor
from bioacoustica.testing.evaluator import Evaluator
print("BioAcoustica testing modules loaded.")

2026-02-24 20:10:41.601817: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 20:10:41.638845: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 20:10:42.176858: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


BioAcoustica testing modules loaded.


## 2. Running Inference on Test Files

We use the `audio` and `spectrogram` sections of the config to drive the `Predictor`. We strictly target files in `TestingFiles.txt`.

In [2]:
CONFIG_PATH = "../../configs/gibbon.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

AUDIO_DIR = "../../data/raw/audio"
MODEL_PATH = "../../models/custom_cnn_20260224_200208_best.h5"
PRED_DIR = "../../data/predictions"
TEST_LIST = "../../data/metadata/TestingFiles.txt"

predictor = Predictor(
    audio_dir=AUDIO_DIR,
    output_dir=PRED_DIR,
    model_path=MODEL_PATH,
    downsample_rate=config['audio']['sample_rate'],
    lowpass_cutoff=config['audio']['lowpass_cutoff'],
    n_fft=config['spectrogram']['n_fft'],
    hop_length=config['spectrogram']['hop_length'],
    n_mels=config['spectrogram']['n_mels'],
    f_min=config['spectrogram'].get('f_min', 0),
    f_max=config['spectrogram']['f_max']
)

if os.path.exists(TEST_LIST):
    with open(TEST_LIST, "r") as f:
        test_files = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    
    print(f"Running inference on {len(test_files)} test files...")
    for f in test_files:
        # Check if audio exists (with .wav or .WAV)
        audio_file = f + ".wav"
        if not os.path.exists(os.path.join(AUDIO_DIR, audio_file)):
             audio_file = f + ".WAV"
        
        if os.path.exists(os.path.join(AUDIO_DIR, audio_file)):
            print(f"Predicting: {audio_file}")
            predictor.predict_file(audio_file, threshold=0.5)
        else:
            print(f"Warning: Audio file not found for {f}")
else:
    print(f"Error: Testing list not found at {TEST_LIST}")

2026-02-24 20:10:51.824286: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-24 20:10:51.862530: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Running inference on 2 test files...
Predicting: HGSM3C_0+1_20150811_052300.wav


Predicting: HGSM3SOL_0+1_20160305_060000.wav


## 3. Evaluation

Now we compare our predictions against the ground truth annotations for the same test files. This part is synchronized with the CLI `evaluate.py` output.

In [3]:
evaluator = Evaluator(audio_dir=AUDIO_DIR)
GT_DIR = "../../data/raw/annotations"

df_gt_list = []
df_pred_list = []
target_files = test_files # Use the same list as CLI

for base_name in target_files:
    gt_path = os.path.join(GT_DIR, base_name + ".svl")
    pred_path = os.path.join(PRED_DIR, base_name + ".svl")
    
    audio_file = base_name + ".wav"
    if not os.path.exists(os.path.join(AUDIO_DIR, audio_file)):
         audio_file = base_name + ".WAV"
         
    # Ground Truth MUST exist for each file in the testing list
    if not os.path.exists(gt_path):
        print(f"Warning: Ground truth file missing for {base_name}. Skipping from evaluation.")
        continue
        
    df_gt_list.append(evaluator.read_svl(gt_path, audio_file))

    # Prediction might not exist if 0 detections were found
    if os.path.exists(pred_path):
        df_pred_list.append(evaluator.read_svl(pred_path, audio_file))
    else:
        # Add empty DF but with AudioFile info to ensure time tracking is correct
        df_pred_list.append(pd.DataFrame(columns=["Start", "End", "Label", "StartSec", "EndSec", "AudioFile"]))

if df_gt_list:
    df_gt = pd.concat(df_gt_list)
    df_pred = pd.concat(df_pred_list) if df_pred_list else pd.DataFrame(columns=["Start", "End", "Label", "StartSec", "EndSec", "AudioFile"])

    results = evaluator.evaluate(
        df_gt=df_gt, 
        df_pred=df_pred, 
        target_label=config['species_name'],
        min_overlap_pct=0.5,
        test_files=target_files
    )
    
    print("\n" + "="*45)
    print(" EVALUATED FILES ")
    print("="*45)
    for f in target_files:
        print(f" - {f}")

    print("\n" + "="*45)
    print(" BIOACOUSTICA EVALUATION RESULTS ")
    print("="*45)
    print(f"Target Species:     {config['species_name']}")
    print(f"Ground Truth Events:{results['NumGT']}")
    print(f"Merged Detections:  {results['NumDetections']}")
    print(f"True Positives:     {results['TP']}")
    print(f"False Positives:    {results['FP']}")
    print(f"False Negatives:    {results['FN']}")
    print("-" * 45)
    print(f"Precision:          {results['Precision']:.4f}")
    print(f"Recall:             {results['Recall']:.4f}")
    print(f"F1-Score:           {results['F1']:.4f}")
    print(f"False Alarms / Hr:  {results['FP_per_hour']:.2f}")
    print(f"Analyzed Hours:     {results['TotalHours']:.2f}")
    print("="*45)
else:
    print("No matching GT/Prediction files found for evaluation.")


 EVALUATED FILES 
 - HGSM3C_0+1_20150811_052300
 - HGSM3SOL_0+1_20160305_060000

 BIOACOUSTICA EVALUATION RESULTS 
Target Species:     gibbon
Ground Truth Events:151
Merged Detections:  134
True Positives:     122
False Positives:    19
False Negatives:    29
---------------------------------------------
Precision:          0.8582
Recall:             0.8079
F1-Score:           0.8323
False Alarms / Hr:  5.97
Analyzed Hours:     3.18


## 4. Results Interpretation

The evaluation results provide a research-grade assessment of the model's performance on the unseen test set:

- **Precision**: Shows how many of our detections were actually gibbons. High precision means fewer false alarms.
- **Recall**: Shows how many of the actual gibbon calls we captured. High recall means we aren't missing many calls.
- **FP / Hr**: The number of false alarms per hour of audio. This is a critical metric for long-term monitoring efficiency.
- **Analyzed Hours**: The total duration of the test set files. This confirms that the evaluation is correctly normalized across the entire testing subset defined in `TestingFiles.txt`.